In [0]:
spark.conf.set(
    "fs.azure.account.key.shadowrevenuedatalake.dfs.core.windows.net",
    "VKLrAB5qdfVdhbicaVVEA3i14wezBaDAaGDj4cXpZ8BdiuJapOyjR6u4gJHtDTjueSvBtpxzqzln+AStoxfJsQ=="
)

In [0]:
orders = spark.read.format("delta").load("abfss://bronze@shadowrevenuedatalake.dfs.core.windows.net/delta/orders")
payments = spark.read.format("delta").load("abfss://bronze@shadowrevenuedatalake.dfs.core.windows.net/delta/payments")
products = spark.read.format("delta").load("abfss://bronze@shadowrevenuedatalake.dfs.core.windows.net/delta/products")
customers = spark.read.format("delta").load("abfss://bronze@shadowrevenuedatalake.dfs.core.windows.net/delta/customers")

In [0]:
orders.createOrReplaceTempView("silver_orders_bad")
payments.createOrReplaceTempView("silver_payments_bad")
products.filter("is_current = 1").createOrReplaceTempView("silver_products_bad")
customers.createOrReplaceTempView("silver_customers_bad")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, to_date
from pyspark.sql.types import DecimalType

window_spec = Window.partitionBy("order_id").orderBy(col("order_date").desc())

orders_clean = (
    orders
    .withColumn("rn", row_number().over(window_spec))
    .filter("rn = 1")
    .drop("rn")
    .withColumn("price", col("price").cast(DecimalType(10,4)))
    .withColumn("order_date", to_date("order_date"))
    .filter(col("order_id").isNotNull())
)

print("Before dedup:", orders.count(), "| After dedup:", orders_clean.count())

Before dedup: 20400 | After dedup: 20000


In [0]:
payments_clean = (
    payments
    .withColumn("payment_amount", col("payment_amount").cast(DecimalType(10,4)))
    .withColumn("payment_date", to_date("payment_date"))
    .filter(col("order_id").isNotNull())
)

products_clean = (
    products
    .filter("is_current = 1")
    .withColumn("price", col("price").cast(DecimalType(10,4)))
    .withColumn("effective_date", to_date("effective_date"))
)

In [0]:
orders_clean.createOrReplaceTempView("silver_orders_good")
payments_clean.createOrReplaceTempView("silver_payments_good")
products_clean.createOrReplaceTempView("silver_products_good")
customers.createOrReplaceTempView("silver_customers_good")

orders_clean.write.format("delta").mode("overwrite").save("abfss://silver@shadowrevenuedatalake.dfs.core.windows.net/orders_good")
payments_clean.write.format("delta").mode("overwrite").save("abfss://silver@shadowrevenuedatalake.dfs.core.windows.net/payments_good")
products_clean.write.format("delta").mode("overwrite").save("abfss://silver@shadowrevenuedatalake.dfs.core.windows.net/products_good")
customers.write.format("delta").mode("overwrite").save("abfss://silver@shadowrevenuedatalake.dfs.core.windows.net/customers_good")